Link - http://platform.stratascratch.com/coding/10172-best-selling-item?code_type=6

In [0]:
import pyspark.sql.types as T
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, DateType

online_retail_schema = StructType([
    StructField("invoiceno", StringType(), True),
    StructField("stockcode", StringType(), True),
    StructField("description", StringType(), True),
    StructField("quantity", LongType(), True),
    StructField("invoicedate", DateType(), True),
    StructField("unitprice", DoubleType(), True),
    StructField("customerid", DoubleType(), True),
    StructField("country", StringType(), True)
])

In [0]:
from datetime import date

online_retail_data = [

    # 🔹 JANUARY (tie case after aggregation)
    ("10001", "S1", "T-Shirt", 10, date(2019, 1, 5), 10.0, 101.0, "UK"),   # 100
    ("10002", "S2", "Shoes", 5, date(2019, 1, 10), 20.0, 102.0, "UK"),     # 100 (tie)

    ("10003", "S1", "T-Shirt", 5, date(2019, 1, 15), 10.0, 103.0, "UK"),   # total T-Shirt = 150

    # 🔹 FEBRUARY (clear winner)
    ("10004", "S3", "Jacket", 3, date(2019, 2, 1), 50.0, 104.0, "France"), # 150
    ("10005", "S4", "Hat", 10, date(2019, 2, 2), 5.0, 105.0, "France"),    # 50

    # 🔹 FEBRUARY (returns - should be ignored)
    ("C10006", "S3", "Jacket", -2, date(2019, 2, 3), 50.0, 104.0, "France"),

    # 🔹 MARCH (multiple rows same product)
    ("10007", "S5", "Laptop", 1, date(2019, 3, 1), 1000.0, 106.0, "Germany"), # 1000
    ("10008", "S5", "Laptop", 1, date(2019, 3, 5), 1000.0, 107.0, "Germany"), # total = 2000

    ("10009", "S6", "Phone", 5, date(2019, 3, 10), 300.0, 108.0, "Germany"), # 1500

    # 🔹 APRIL (tie case)
    ("10010", "S7", "Tablet", 2, date(2019, 4, 1), 500.0, 109.0, "India"), # 1000
    ("10011", "S8", "Monitor", 4, date(2019, 4, 3), 250.0, 110.0, "India"), # 1000 (tie)

    # 🔹 APRIL (returns)
    ("C10012", "S7", "Tablet", -1, date(2019, 4, 5), 500.0, 109.0, "India"),

    # 🔹 MAY (outside if you filter later)
    ("10013", "S9", "Keyboard", 10, date(2019, 5, 1), 50.0, 111.0, "USA"), # 500
]

In [0]:
online_retail_df = spark.createDataFrame(online_retail_data, online_retail_schema)

In [0]:
online_retail_df.show(10)

In [0]:
df_1 = online_retail_df.filter(F.col("quantity") > 0 )\
                       .withColumn("total_price", F.col("unitprice")*F.col("quantity"))
df_1.show()

In [0]:
w = Window.partitionBy(F.month("invoicedate")).orderBy(F.desc("total_price"))
df_2 = df_1.withColumn("rank", F.dense_rank().over(w))
df_2.show()

In [0]:
df_3 = df_2.filter(F.col("rank") == 1)
df_3.show()

In [0]:
df_4 = df_2.withColumn("month", F.month("invoicedate"))
df_5 = df_4.select("month","description","total_price")\
            .filter(F.col("rank") == 1)
df_5.show()